# LeetCode 1733 — Minimum Number of People to Teach bằng Python

Link đề: https://leetcode.com/problems/minimum-number-of-people-to-teach/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Greedy

Skeleton LeetCode:

```python
class Solution:
    def minimumTeachings(self, n: int, languages: List[List[int]],
                         friendships: List[List[int]]) -> int:
        ...
```

Lộ trình: **vét cạn** (thử mọi ngôn ngữ × mọi tập người được dạy, `m` nhỏ) → nhận ra với ngôn ngữ cố định, tập người **bắt buộc** phải dạy được xác định duy nhất → **thử từng ngôn ngữ**, gom người cần dạy (`O(n · F)`) → **đếm một lượt**: chỉ những người nằm trong "cặp bạn không nói chuyện được" mới quan trọng; đáp án = `|S| − max_L #{u ∈ S biết L}` → biến thể **bitmask**.

## 1. Mục tiêu học

1. Đọc kỹ ràng buộc "chọn **một** ngôn ngữ và dạy nó cho một số người" — bài toán tối ưu trên **một tham số rời rạc** (ngôn ngữ) cộng phần đếm.
2. Xác định **cặp bạn "hỏng"** (không có ngôn ngữ chung) bằng giao tập hợp; hiểu rằng cặp đã nói chuyện được thì không cần quan tâm.
3. Chứng minh: với ngôn ngữ `L` cố định, tập người cần dạy **tối thiểu** = những người trong cặp hỏng mà chưa biết `L`.
4. Tối ưu bằng **đếm tần suất** ngôn ngữ trong tập người liên quan `S` ⇒ đáp án `|S| − max count`.
5. Dùng **bitmask** (`int` Python) để giao tập ngôn ngữ nhanh.

## 2. Đề bài & ví dụ

Một mạng xã hội có `m` người dùng và một số quan hệ bạn bè. Hai người **nói chuyện được** nếu họ biết **chung ít nhất một** ngôn ngữ.

Cho số nguyên `n`, mảng `languages` và mảng `friendships`:
- Có `n` ngôn ngữ đánh số `1..n`.
- `languages[i]` là tập ngôn ngữ người thứ `i + 1` biết (người đánh số từ 1).
- `friendships[i] = [u_i, v_i]` nghĩa là `u_i` và `v_i` là bạn.

Bạn được chọn **một** ngôn ngữ và dạy nó cho một số người để **mọi cặp bạn** đều nói chuyện được. Trả về **số người ít nhất** cần dạy.

Lưu ý: tình bạn **không bắc cầu** (x bạn y, y bạn z không có nghĩa x bạn z).

**Ví dụ 1:** `n = 2`, `languages = [[1],[2],[1,2]]`, `friendships = [[1,2],[1,3],[2,3]]` → `1`. Dạy người 1 ngôn ngữ 2, hoặc dạy người 2 ngôn ngữ 1.

**Ví dụ 2:** `n = 3`, `languages = [[2],[1,3],[1,2],[3]]`, `friendships = [[1,4],[1,2],[3,4],[2,3]]` → `2`. Dạy ngôn ngữ 3 cho người 1 và 3.

**Ràng buộc:**
- `2 ≤ n ≤ 500`, `1 ≤ m = languages.length ≤ 500`
- `1 ≤ languages[i].length ≤ n`, `1 ≤ languages[i][j] ≤ n`, giá trị trong mỗi `languages[i]` phân biệt
- `1 ≤ friendships.length ≤ 500`, `1 ≤ u_i < v_i ≤ m`, các cặp phân biệt

In [1]:
import random
import time
from collections import Counter
from itertools import combinations
from typing import List

examples = [
    ((2, [[1], [2], [1, 2]], [[1, 2], [1, 3], [2, 3]]), 1),
    ((3, [[2], [1, 3], [1, 2], [3]], [[1, 4], [1, 2], [3, 4], [2, 3]]), 2),
]
for (n, langs, fr), want in examples:
    print(f"n={n}, languages={langs}, friendships={fr} -> kỳ vọng {want}")

n=2, languages=[[1], [2], [1, 2]], friendships=[[1, 2], [1, 3], [2, 3]] -> kỳ vọng 1
n=3, languages=[[2], [1, 3], [1, 2], [3]], friendships=[[1, 4], [1, 2], [3, 4], [2, 3]] -> kỳ vọng 2


## 3. Kiến thức nền

### 3.1 Tập hợp và giao tập hợp

`set(a) & set(b)` là giao; `isdisjoint` kiểm tra rỗng nhanh hơn (dừng sớm, không tạo tập mới).

In [2]:
a, b = {1, 3}, {2, 3}
print(a & b, a.isdisjoint(b), {1}.isdisjoint({2}))

{3} False True


### 3.2 Chỉ số từ 1 sang từ 0

Người được đánh số `1..m` trong `friendships` nhưng `languages` dùng chỉ số `0..m−1` ⇒ `languages[u − 1]`.

### 3.3 `Counter` đếm tần suất

In [3]:
c = Counter()
for user_langs in [[1, 2], [2], [2, 3]]:
    c.update(user_langs)
print(c, c.most_common(1))

Counter({2: 3, 1: 1, 3: 1}) [(2, 3)]


### 3.4 Bitmask biểu diễn tập ngôn ngữ

Tập `{1, 3}` ⇔ số `0b1010` (bit `L` bật nếu biết ngôn ngữ `L`). Giao khác rỗng ⇔ `mask_u & mask_v != 0`. Python `int` không giới hạn số bit nên `n = 500` vẫn ổn.

In [4]:
def to_mask(langs):
    m = 0
    for L in langs:
        m |= 1 << L
    return m
print(bin(to_mask([1, 3])), to_mask([1, 3]) & to_mask([2, 3]) != 0, to_mask([1]) & to_mask([2]) != 0)

0b1010 True False


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 2: `languages = [[2],[1,3],[1,2],[3]]` (người 1..4).

| cặp bạn | ngôn ngữ người u | ngôn ngữ người v | chung? |
|---|---|---|---|
| (1,4) | {2} | {3} | **không** |
| (1,2) | {2} | {1,3} | **không** |
| (3,4) | {1,2} | {3} | **không** |
| (2,3) | {1,3} | {1,2} | có (1) |

Cặp hỏng: (1,4), (1,2), (3,4) ⇒ tập người liên quan `S = {1, 2, 3, 4}`.

Thử từng ngôn ngữ `L`: phải dạy mọi người trong `S` chưa biết `L` (để **mọi** cặp hỏng đều có `L` chung):

| L | ai trong S đã biết L | cần dạy |
|---|---|---|
| 1 | {2, 3} | {1, 4} ⇒ 2 |
| 2 | {1, 3} | {2, 4} ⇒ 2 |
| 3 | {2, 4} | {1, 3} ⇒ 2 |

Đáp án `2`.

## 5. Cách 1 — Brute Force / baseline: thử mọi ngôn ngữ × mọi tập người

**Ý tưởng:** với mỗi ngôn ngữ `L` và mỗi tập con `T` người (theo kích thước tăng dần), dạy `L` cho `T` rồi kiểm tra mọi cặp bạn nói chuyện được. Không dùng bất kỳ nhận xét nào — chỉ bám định nghĩa. Chỉ dùng cho `m ≤ 10`.

In [5]:
def all_ok(langs_sets, friendships):
    return all(not langs_sets[u - 1].isdisjoint(langs_sets[v - 1]) for u, v in friendships)

def min_teach_brute(n, languages, friendships):
    m = len(languages)
    base = [set(l) for l in languages]
    for k in range(m + 1):
        for L in range(1, n + 1):
            for T in combinations(range(m), k):
                cur = [s | {L} if i in T else s for i, s in enumerate(base)]
                if all_ok(cur, friendships):
                    return k
    return m

for (n, langs, fr), want in examples:
    print(min_teach_brute(n, langs, fr), "kỳ vọng", want)

1 kỳ vọng 1
2 kỳ vọng 2


**Độ phức tạp:** `O(n · 2^m · F · Σ)` — mũ theo `m`. Không dùng được cho `m = 500`.

**Nút thắt:** với `L` cố định, ta không cần thử tập con — tập người **bắt buộc** phải dạy đã xác định duy nhất (mục 6).

In [6]:
random.seed(0)
for m in [6, 8, 10]:
    n = 4
    langs = [random.sample(range(1, n + 1), random.randint(1, 2)) for _ in range(m)]
    fr = random.sample([[u, v] for u in range(1, m + 1) for v in range(u + 1, m + 1)], m)
    t0 = time.perf_counter(); r = min_teach_brute(n, langs, fr)
    print(f"m={m}: {r}, {time.perf_counter() - t0:.3f}s")

m=6: 0, 0.000s
m=8: 1, 0.000s
m=10: 3, 0.001s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — cặp tốt không cần quan tâm.** Dạy thêm ngôn ngữ không bao giờ làm một cặp đang nói chuyện được trở nên không được. Chỉ **cặp hỏng** (không có ngôn ngữ chung) cần xử lý.

**Insight 2 — với ngôn ngữ `L` cố định, tập dạy tối thiểu là duy nhất.** Ta chỉ dạy một ngôn ngữ `L`. Một cặp hỏng `(u, v)` hiện không có ngôn ngữ chung; sau khi dạy, ngôn ngữ chung (nếu có) **chỉ có thể là `L`** (ngôn ngữ khác không thay đổi). Vậy **cả** `u` và `v` phải biết `L` ⇒ mọi người trong cặp hỏng chưa biết `L` **bắt buộc** phải học. Và dạy đúng những người đó là đủ.

⇒ Gọi `S` = tập mọi người xuất hiện trong ít nhất một cặp hỏng. Số người cần dạy với `L` là `|{u ∈ S : u không biết L}| = |S| − cnt[L]`, với `cnt[L]` = số người trong `S` biết `L`.

**Đáp án:** `|S| − max_L cnt[L]` (nếu không có cặp hỏng thì `S` rỗng ⇒ `0`).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Thử từng ngôn ngữ, gom người cần dạy (`O(n · F)`)

In [7]:
def min_teach_per_language(n, languages, friendships):
    sets = [set(l) for l in languages]
    bad = [(u - 1, v - 1) for u, v in friendships if sets[u - 1].isdisjoint(sets[v - 1])]
    if not bad:
        return 0
    best = len(languages)
    for L in range(1, n + 1):
        need = set()
        for u, v in bad:
            if L not in sets[u]:
                need.add(u)
            if L not in sets[v]:
                need.add(v)
        best = min(best, len(need))
    return best

for (n, langs, fr), want in examples:
    print(min_teach_per_language(n, langs, fr), "kỳ vọng", want)

1 kỳ vọng 1
2 kỳ vọng 2


**Độ phức tạp:** tìm cặp hỏng `O(F · ℓ)` (`ℓ` = độ dài tập ngôn ngữ); vòng chính `O(n · F)` ⇒ `500 · 500 = 2.5·10⁵` — rất nhanh. Space `O(m + Σ|languages|)`.

### 7.2 Cách 3 — Đếm một lượt trên tập `S`

Từng đoạn:
1. Tìm cặp hỏng, gom người vào `S` (set chỉ số).
2. `cnt` = `Counter` các ngôn ngữ của những người trong `S`.
3. Trả về `len(S) − max(cnt.values())` (hoặc `0` nếu `S` rỗng).

In [8]:
def min_teach_count(n, languages, friendships):
    sets = [set(l) for l in languages]
    S = set()
    for u, v in friendships:
        if sets[u - 1].isdisjoint(sets[v - 1]):
            S.add(u - 1)
            S.add(v - 1)
    if not S:
        return 0
    cnt = Counter()
    for u in S:
        cnt.update(languages[u])
    return len(S) - max(cnt.values())

for (n, langs, fr), want in examples:
    print(min_teach_count(n, langs, fr), "kỳ vọng", want)

1 kỳ vọng 1
2 kỳ vọng 2


**Độ phức tạp:** Time `O(F · ℓ + Σ_{u∈S} |languages[u]|)`, Space `O(m + n)`.

Lưu ý: `max(cnt.values())` — nếu một ngôn ngữ không ai trong `S` biết thì `cnt` không có khóa đó (đếm 0) — không ảnh hưởng vì ta lấy max. `cnt` không rỗng vì mỗi người biết ít nhất một ngôn ngữ.

### 7.3 Cách 4 — Bitmask cho tập ngôn ngữ

Mỗi người một số nguyên `mask`; kiểm tra cặp hỏng bằng `mask_u & mask_v == 0` (một phép AND trên số lớn thay vì giao set). Đếm ngôn ngữ vẫn bằng mảng `cnt[1..n]`.

In [9]:
def min_teach_bitmask(n, languages, friendships):
    masks = []
    for langs in languages:
        mk = 0
        for L in langs:
            mk |= 1 << L
        masks.append(mk)
    in_S = [False] * len(languages)
    for u, v in friendships:
        if masks[u - 1] & masks[v - 1] == 0:
            in_S[u - 1] = in_S[v - 1] = True
    size = sum(in_S)
    if size == 0:
        return 0
    cnt = [0] * (n + 1)
    for u, flag in enumerate(in_S):
        if flag:
            for L in languages[u]:
                cnt[L] += 1
    return size - max(cnt)

for (n, langs, fr), want in examples:
    print(min_teach_bitmask(n, langs, fr), "kỳ vọng", want)

1 kỳ vọng 1
2 kỳ vọng 2


**Độ phức tạp:** Time `O(Σ|languages| + F · n/64)` (AND số `n` bit), Space `O(m · n/64 + n)`.

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ 1 và ví dụ 2:

In [10]:
def trace_count(n, languages, friendships):
    sets = [set(l) for l in languages]; S = set()
    for u, v in friendships:
        common = sets[u - 1] & sets[v - 1]
        if not common:
            S |= {u, v}
        print(f"cặp ({u},{v}): {sorted(sets[u - 1])} vs {sorted(sets[v - 1])} -> chung {sorted(common) or 'KHÔNG'}")
    print("S (người trong cặp hỏng) =", sorted(S))
    if not S:
        print("không có cặp hỏng -> 0"); return 0
    cnt = Counter()
    for u in S:
        cnt.update(languages[u - 1])
    for L in range(1, n + 1):
        print(f"  ngôn ngữ {L}: {cnt[L]} người trong S đã biết -> cần dạy {len(S) - cnt[L]}")
    ans = len(S) - max(cnt.values())
    print("đáp án =", ans)
    return ans

trace_count(*examples[0][0])
print()
trace_count(*examples[1][0])

cặp (1,2): [1] vs [2] -> chung KHÔNG
cặp (1,3): [1] vs [1, 2] -> chung [1]
cặp (2,3): [2] vs [1, 2] -> chung [2]
S (người trong cặp hỏng) = [1, 2]
  ngôn ngữ 1: 1 người trong S đã biết -> cần dạy 1
  ngôn ngữ 2: 1 người trong S đã biết -> cần dạy 1
đáp án = 1

cặp (1,4): [2] vs [3] -> chung KHÔNG
cặp (1,2): [2] vs [1, 3] -> chung KHÔNG
cặp (3,4): [1, 2] vs [3] -> chung KHÔNG
cặp (2,3): [1, 3] vs [1, 2] -> chung [1]
S (người trong cặp hỏng) = [1, 2, 3, 4]
  ngôn ngữ 1: 2 người trong S đã biết -> cần dạy 2
  ngôn ngữ 2: 2 người trong S đã biết -> cần dạy 2
  ngôn ngữ 3: 2 người trong S đã biết -> cần dạy 2
đáp án = 2


2

## 9. Code nộp LeetCode

In [11]:
class Solution:
    def minimumTeachings(self, n: int, languages: List[List[int]],
                         friendships: List[List[int]]) -> int:
        sets = [set(l) for l in languages]
        S = set()
        for u, v in friendships:
            if sets[u - 1].isdisjoint(sets[v - 1]):
                S.add(u - 1)
                S.add(v - 1)
        if not S:
            return 0
        cnt = Counter()
        for u in S:
            cnt.update(languages[u])
        return len(S) - max(cnt.values())

sol = Solution()
for (n, langs, fr), want in examples:
    got = sol.minimumTeachings(n, langs, fr)
    print(got, "OK" if got == want else "SAI")

1 OK
2 OK


## 10. Vì sao thuật toán đúng

Cố định ngôn ngữ dạy `L`, gọi `T` là tập người được dạy. Điều kiện thành công: với mọi cặp bạn `(u, v)`, sau khi dạy họ có ngôn ngữ chung.

**(a) Điều kiện cần.** Xét cặp hỏng `(u, v)` (ban đầu không có ngôn ngữ chung). Sau khi dạy, tập của `u` là `A_u ∪ ({L} nếu u ∈ T)`, tương tự `v`. Giao mới ⊆ `(A_u ∩ A_v) ∪ {L} = {L}`. Vậy giao khác rỗng ⇔ `L` thuộc cả hai ⇔ (`L ∈ A_u` hoặc `u ∈ T`) và (`L ∈ A_v` hoặc `v ∈ T`). ⇒ mọi `u ∈ S` không biết `L` phải thuộc `T`.

**(b) Điều kiện đủ.** Chọn `T_L = {u ∈ S : L ∉ A_u}`. Mọi cặp hỏng khi đó đều có `L` chung; mọi cặp tốt vẫn tốt (dạy thêm không làm mất ngôn ngữ). ✓

Vậy với mỗi `L`, tối ưu là `|T_L| = |S| − cnt[L]`; đáp án là min theo `L` = `|S| − max_L cnt[L]`. Nếu không có cặp hỏng, không cần dạy ai.

Mục 12 kiểm chứng với vét cạn tập con.

## 11. Phân tích Time / Space

Gọi `m ≤ 500` số người, `n ≤ 500` số ngôn ngữ, `F ≤ 500` số cặp bạn, `ℓ` độ dài trung bình `languages[i]`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Vét cạn tập con | `O(n · 2^m · F · ℓ)` | `O(m · ℓ)` | `m ≤ 10` |
| 2. Thử từng ngôn ngữ | `O(F·ℓ + n·F)` | `O(m·ℓ)` | rất nhanh |
| 3. Đếm trên `S` | `O(F·ℓ + Σ_S ℓ)` | `O(m·ℓ + n)` | **nộp bài** |
| 4. Bitmask | `O(Σℓ + F·n/64)` | `O(m·n/64)` | giao nhanh |

In [12]:
random.seed(1)
n, m = 500, 500
langs = [random.sample(range(1, n + 1), random.randint(1, 5)) for _ in range(m)]
pairs = set()
while len(pairs) < 500:
    u, v = sorted(random.sample(range(1, m + 1), 2)); pairs.add((u, v))
fr = [list(p) for p in pairs]
for name, fn in [("per-lang", min_teach_per_language), ("count", min_teach_count),
                 ("bitmask", min_teach_bitmask), ("submit", Solution().minimumTeachings)]:
    t0 = time.perf_counter(); r = fn(n, langs, fr)
    print(f"{name:8s}: {r}  {time.perf_counter() - t0:.4f}s")

per-lang: 427  0.0232s
count   : 427  0.0004s
bitmask : 427  0.0003s
submit  : 427  0.0003s


## 12. Bộ test

Sinh input đúng ràng buộc: mỗi người biết 1..n ngôn ngữ phân biệt, các cặp `u < v` phân biệt.

In [13]:
def gen(rng, n_max=4, m_max=7):
    n = rng.randint(2, n_max); m = rng.randint(2, m_max)
    langs = [rng.sample(range(1, n + 1), rng.randint(1, n)) for _ in range(m)]
    all_pairs = [[u, v] for u in range(1, m + 1) for v in range(u + 1, m + 1)]
    fr = rng.sample(all_pairs, rng.randint(1, len(all_pairs)))
    return n, langs, fr

fns = [min_teach_per_language, min_teach_count, min_teach_bitmask, lambda a, b, c: Solution().minimumTeachings(a, b, c)]

for (n, langs, fr), want in examples:
    for fn in fns:
        assert fn(n, langs, fr) == want

edge = [
    ((2, [[1], [1]], [[1, 2]]), 0),                    # đã nói chuyện được
    ((2, [[1], [2]], [[1, 2]]), 1),
    ((3, [[1], [2], [3]], [[1, 2], [2, 3], [1, 3]]), 2),
    ((2, [[1], [2], [1], [2]], [[1, 2], [3, 4]]), 2),
]
for (n, langs, fr), want in edge:
    assert min_teach_brute(n, langs, fr) == want, (n, langs, fr)
    for fn in fns:
        assert fn(n, langs, fr) == want, (fn, n, langs, fr)

rng = random.Random(2024)
for _ in range(400):
    n, langs, fr = gen(rng)
    want = min_teach_brute(n, langs, fr)
    for fn in fns:
        assert fn(n, langs, fr) == want, (fn, n, langs, fr)
print("ví dụ + edge + 400 bộ ngẫu nhiên khớp vét cạn tập con")

ví dụ + edge + 400 bộ ngẫu nhiên khớp vét cạn tập con


In [14]:
rng = random.Random(7)
for _ in range(100):
    n, langs, fr = gen(rng, n_max=30, m_max=40)
    r = min_teach_per_language(n, langs, fr)
    assert r == min_teach_count(n, langs, fr) == min_teach_bitmask(n, langs, fr)
print("bộ vừa (n ≤ 30, m ≤ 40): các cách nhanh khớp nhau")

bộ vừa (n ≤ 30, m ≤ 40): các cách nhanh khớp nhau


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Đếm ngôn ngữ trên **tất cả** người dùng, không chỉ tập `S`

Người không nằm trong cặp hỏng không cần học ⇒ không nên ảnh hưởng tới việc chọn ngôn ngữ.

In [15]:
def wrong_all_users(n, languages, friendships):
    sets = [set(l) for l in languages]; S = set()
    for u, v in friendships:
        if sets[u - 1].isdisjoint(sets[v - 1]):
            S |= {u - 1, v - 1}
    if not S:
        return 0
    cnt = Counter()
    for langs in languages:               # SAI: đếm trên mọi người
        cnt.update(langs)
    L = cnt.most_common(1)[0][0]
    return sum(1 for u in S if L not in sets[u])

rng = random.Random(3)
for _ in range(3000):
    n, langs, fr = gen(rng)
    if wrong_all_users(n, langs, fr) != min_teach_count(n, langs, fr):
        print("phản ví dụ:", n, langs, fr, "| sai =", wrong_all_users(n, langs, fr), "| đúng =", min_teach_count(n, langs, fr))
        break

phản ví dụ: 3 [[3], [1, 3], [2, 3], [1, 3], [2], [2, 3, 1], [2]] [[3, 7], [4, 5], [4, 6], [4, 7], [2, 4], [3, 4], [3, 6], [1, 6], [6, 7], [1, 2]] | sai = 2 | đúng = 1


**Sửa:** chỉ đếm ngôn ngữ của người trong `S`.

### Lỗi 2 — Dạy cả hai người của **mọi** cặp hỏng, đếm trùng

Cộng `2` (hoặc số người chưa biết `L`) cho mỗi cặp hỏng ⇒ một người thuộc nhiều cặp bị đếm nhiều lần.

In [16]:
def wrong_count_per_pair(n, languages, friendships):
    sets = [set(l) for l in languages]
    bad = [(u - 1, v - 1) for u, v in friendships if sets[u - 1].isdisjoint(sets[v - 1])]
    if not bad:
        return 0
    return min(sum((L not in sets[u]) + (L not in sets[v]) for u, v in bad) for L in range(1, n + 1))   # SAI: đếm trùng

for (n, langs, fr), want in examples:
    print("sai =", wrong_count_per_pair(n, langs, fr), "| đúng =", want)

sai = 1 | đúng = 1
sai = 3 | đúng = 2


**Sửa:** gom vào **tập** `need` (Cách 2) để mỗi người chỉ đếm một lần.

### Lỗi 3 — Nghĩ rằng tình bạn bắc cầu (gom thành thành phần liên thông)

Dạy theo thành phần liên thông của đồ thị bạn bè ⇒ dạy cả những người không cần.

In [17]:
def wrong_transitive(n, languages, friendships):
    m = len(languages); parent = list(range(m))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    sets = [set(l) for l in languages]
    for u, v in friendships:
        if sets[u - 1].isdisjoint(sets[v - 1]):
            parent[find(u - 1)] = find(v - 1)
    # SAI: coi mọi người trong thành phần chứa cặp hỏng đều "liên quan", kể cả người chỉ có cặp tốt
    for u, v in friendships:
        parent[find(u - 1)] = find(v - 1)
    roots_bad = {find(u - 1) for u, v in friendships if sets[u - 1].isdisjoint(sets[v - 1])}
    S = [u for u in range(m) if find(u) in roots_bad]
    if not S:
        return 0
    cnt = Counter()
    for u in S:
        cnt.update(languages[u])
    return len(S) - max(cnt.values())

n, langs, fr = 3, [[1], [2], [2, 3], [3]], [[1, 2], [2, 3], [3, 4]]
print("sai =", wrong_transitive(n, langs, fr), "| đúng =", min_teach_count(n, langs, fr))

sai = 2 | đúng = 1


Người 3 và 4 chỉ có cặp tốt nhưng bị kéo vào vì cùng thành phần. **Sửa:** chỉ lấy hai đầu của các **cặp hỏng**.

### Lỗi 4 — Dùng chỉ số người từ 1 trực tiếp vào `languages`

`languages[u]` thay vì `languages[u − 1]` ⇒ lệch người, có thể `IndexError` với `u = m`.

In [18]:
def wrong_index(n, languages, friendships):
    sets = [set(l) for l in languages]; S = set()
    for u, v in friendships:
        su = sets[u] if u < len(sets) else set()      # SAI: phải là u - 1 (ở đây chặn IndexError để minh họa)
        sv = sets[v] if v < len(sets) else set()
        if su.isdisjoint(sv):
            S |= {u, v}
    return len(S)

for (n, langs, fr), want in examples:
    print("sai =", wrong_index(n, langs, fr), "| đúng =", want)

sai = 3 | đúng = 1
sai = 4 | đúng = 2


**Sửa:** luôn đổi `u − 1`, `v − 1` khi tra `languages`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chọn **một** giá trị trong miền nhỏ (`n ≤ 500`) rồi tối ưu phần còn lại" ⇒ **thử mọi giá trị**, mỗi giá trị tính nhanh.
- Lọc trước các phần tử **không liên quan** (cặp tốt) — thường đơn giản hóa bài rất nhiều.
- "Phải dạy" vs "nên dạy": tìm tập **bắt buộc** (điều kiện cần) rồi kiểm tra nó **đủ**.
- Đọc kỹ "not transitive".
- Giao tập lớn nhiều lần ⇒ cân nhắc bitmask.

## 15. Biến thể của bài

1. **Được dạy tối đa `k` ngôn ngữ khác nhau:** khó hơn nhiều (bài toán phủ — NP-khó nói chung khi `k` tùy ý).
2. **Trả về ngôn ngữ nên dạy và danh sách người:** lấy `L* = argmax cnt`, người cần dạy `{u ∈ S : L* ∉ A_u}`.
3. **Chi phí dạy mỗi người khác nhau `cost[u]`:** đáp án `min_L Σ_{u ∈ S, L ∉ A_u} cost[u]`.
4. **Tình bạn bắc cầu** (mọi người trong cùng thành phần phải nói chuyện được với nhau): mỗi thành phần phải có một ngôn ngữ chung — đổi tập `S` theo thành phần.

In [19]:
def teaching_plan(n, languages, friendships):
    """Biến thể 2: (số người, ngôn ngữ, danh sách người cần dạy — đánh số từ 1)."""
    sets = [set(l) for l in languages]; S = set()
    for u, v in friendships:
        if sets[u - 1].isdisjoint(sets[v - 1]):
            S |= {u - 1, v - 1}
    if not S:
        return 0, None, []
    cnt = Counter()
    for u in S:
        cnt.update(languages[u])
    L = max(range(1, n + 1), key=lambda x: cnt[x])
    who = sorted(u + 1 for u in S if L not in sets[u])
    return len(who), L, who

def weighted_teach(n, languages, friendships, cost):
    """Biến thể 3."""
    sets = [set(l) for l in languages]; S = set()
    for u, v in friendships:
        if sets[u - 1].isdisjoint(sets[v - 1]):
            S |= {u - 1, v - 1}
    if not S:
        return 0
    return min(sum(cost[u] for u in S if L not in sets[u]) for L in range(1, n + 1))

for (n, langs, fr), want in examples:
    print(teaching_plan(n, langs, fr))
print("có chi phí:", weighted_teach(3, [[2], [1, 3], [1, 2], [3]], [[1, 4], [1, 2], [3, 4], [2, 3]], [10, 1, 1, 1]))

(1, 1, [2])
(2, 1, [1, 4])
có chi phí: 2


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | mọi `L` × mọi tập con | mũ | nhỏ | kiểm chứng |
| 2. Từng ngôn ngữ | gom người bắt buộc | `O(nF)` | `O(mℓ)` | dễ giải thích |
| 3. Đếm trên `S` | `|S| − max cnt` | `O(Fℓ + Σℓ)` | `O(mℓ + n)` | **nộp bài** |
| 4. Bitmask | AND số lớn | `O(Σℓ + F n/64)` | `O(m n/64)` | khi `ℓ` lớn |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `n = 2`, `languages = [[1,2],[1],[2]]`, `friendships = [[1,2],[1,3],[2,3]]`?

**Bài 2 (tính tay):** `n = 3`, `languages = [[1],[2],[3],[1,2,3]]`, `friendships = [[1,2],[2,3],[1,4]]`: tìm `S`, bảng `cnt`, đáp án.

**Bài 3 (điền TODO):** hoàn thiện `broken_pairs(languages, friendships)` trả về danh sách cặp hỏng.

**Bài 4 (tự viết lại):** viết `weighted_teach` (biến thể 3) và kiểm chứng bằng vét cạn tập con có trọng số trên input nhỏ.

**Bài 5 (mini challenge):** biến thể **được dạy hai ngôn ngữ** (mỗi người có thể được dạy một hoặc cả hai): tìm số người ít nhất cần dạy (một người học hai ngôn ngữ vẫn tính là 1). Viết vét cạn trên mọi cặp ngôn ngữ + tập con người cho input nhỏ, và một lời giải tốt hơn nếu bạn tìm được. (Biến thể tự đặt.)

In [20]:
def broken_pairs(languages, friendships):
    # TODO: trả về [(u, v), ...] (đánh số từ 1) với languages[u-1], languages[v-1] không giao nhau
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) Cặp (2,3) hỏng: `{1}` vs `{2}`. (2) `S = {2, 3}`. (3) `cnt[1] = cnt[2] = 1` ⇒ đáp án `1`.

**Bài 2:** (1) Cặp (1,2): `{1}` vs `{2}` hỏng; (2,3) hỏng; (1,4) tốt. (2) `S = {1,2,3}`, mỗi ngôn ngữ đếm 1. (3) Đáp án `2`.

**Bài 3:** (1) Duyệt `friendships`. (2) `set(languages[u-1]).isdisjoint(languages[v-1])`. (3) Giữ cặp hỏng.

**Bài 4:** (1) Cùng khung Cách 2/3 nhưng cộng `cost[u]`. (2) Vét cạn: mọi `L`, mọi tập con, tổng chi phí. (3) So sánh trên 200 input ngẫu nhiên.

**Bài 5:** (1) Với cặp ngôn ngữ `(L1, L2)`, mỗi cặp hỏng cần chung `L1` **hoặc** `L2`. (2) Không còn "tập bắt buộc duy nhất" — đây là bài toán khó hơn. (3) Vét cạn trên `m ≤ 7` là đủ để thực nghiệm.

## 19. Đáp án tham khảo

In [21]:
# Bài 1
assert Solution().minimumTeachings(2, [[1, 2], [1], [2]], [[1, 2], [1, 3], [2, 3]]) == 1

# Bài 2
assert Solution().minimumTeachings(3, [[1], [2], [3], [1, 2, 3]], [[1, 2], [2, 3], [1, 4]]) == 2

# Bài 3
def broken_pairs_ans(languages, friendships):
    return [(u, v) for u, v in friendships if set(languages[u - 1]).isdisjoint(languages[v - 1])]
assert broken_pairs_ans([[2], [1, 3], [1, 2], [3]], [[1, 4], [1, 2], [3, 4], [2, 3]]) == [(1, 4), (1, 2), (3, 4)]

# Bài 4
def weighted_brute(n, languages, friendships, cost):
    m = len(languages); base = [set(l) for l in languages]; best = None
    for L in range(1, n + 1):
        for mask in range(1 << m):
            cur = [s | {L} if mask >> i & 1 else s for i, s in enumerate(base)]
            if all_ok(cur, friendships):
                c = sum(cost[i] for i in range(m) if mask >> i & 1)
                best = c if best is None else min(best, c)
    return best

rng = random.Random(11)
for _ in range(200):
    n, langs, fr = gen(rng, n_max=3, m_max=6)
    cost = [rng.randint(1, 5) for _ in langs]
    assert weighted_teach(n, langs, fr, cost) == weighted_brute(n, langs, fr, cost)
print("Bài 1-4: OK")

Bài 1-4: OK


In [22]:
# Bài 5 — được dạy hai ngôn ngữ (vét cạn để thực nghiệm)
def min_teach_two_langs_brute(n, languages, friendships):
    m = len(languages); base = [set(l) for l in languages]; best = m
    for L1 in range(1, n + 1):
        for L2 in range(L1, n + 1):
            # mỗi người: không học / học L1 / học L2 / học cả hai -> 4^m; với m nhỏ
            def rec(i, cur, cnt):
                nonlocal best
                if cnt >= best: return
                if i == m:
                    if all_ok(cur, friendships): best = cnt
                    return
                for add in ({}, {L1}, {L2}, {L1, L2}):
                    rec(i + 1, cur + [base[i] | set(add)], cnt + (1 if add else 0))
            rec(0, [], 0)
    return best

rng = random.Random(12)
diff = 0
for _ in range(150):
    n, langs, fr = gen(rng, n_max=3, m_max=5)
    one = min_teach_count(n, langs, fr); two = min_teach_two_langs_brute(n, langs, fr)
    assert two <= one
    diff += two < one
print(f"Bài 5: trong 150 bộ, dạy được 2 ngôn ngữ giúp giảm số người ở {diff} bộ (luôn ≤ đáp án một ngôn ngữ)")

Bài 5: trong 150 bộ, dạy được 2 ngôn ngữ giúp giảm số người ở 2 bộ (luôn ≤ đáp án một ngôn ngữ)


## 20. Template nhớ nhanh + Checklist

```python
sets = [set(l) for l in languages]
S = set()
for u, v in friendships:                       # chỉ cặp KHÔNG có ngôn ngữ chung
    if sets[u - 1].isdisjoint(sets[v - 1]):
        S.add(u - 1); S.add(v - 1)
if not S: return 0
cnt = Counter()
for u in S: cnt.update(languages[u])           # đếm ngôn ngữ trong S
return len(S) - max(cnt.values())              # dạy ngôn ngữ phổ biến nhất trong S
```

**Checklist trước khi submit:**
- [ ] Chỉ xét **cặp hỏng**; cặp đã nói chuyện được thì bỏ qua.
- [ ] `S` là **tập** (không đếm trùng người).
- [ ] Đếm ngôn ngữ **chỉ trong `S`**.
- [ ] Đổi chỉ số người `u − 1`.
- [ ] Không có cặp hỏng ⇒ `0`.
- [ ] Tình bạn **không bắc cầu**.